# Cross-camera GAP LeJEPA for cotton boll detection

This notebook **only configures and calls** functions from the `lejepa_cotton` package; no functions are defined here.

1. **Pretraining** - an empty-weight `yolov8n.yaml` backbone is trained with LeJEPA on synchronised frames from cameras 1, 2 and 4. Every scale (P3, P4, P5) is globally average pooled and projected; the prediction loss pulls each camera's embedding toward the mean of the *other* cameras of the same frame, and SIGReg keeps the embeddings isotropic Gaussian.
2. **Evaluation** - the LeJEPA backbone is compared with COCO `yolov8n.pt` weights on (a) cotton-boll detection fine-tuning and (b) a frozen-backbone GAP linear probe for plot status.
3. **Visualization** - loss curves, camera-coloured PCA of embeddings, detection curves/overlays, confusion matrices and metric bars.

## 0. Install the package (run once)

In [3]:
# Run once from this notebook; `..` is the repository root that holds setup.py.
%pip install -e lejepa_cotton_gap

ERROR: lejepa_cotton_gap is not a valid editable requirement. It should either be a path to a local project or a VCS URL (beginning with bzr+http, bzr+https, bzr+ssh, bzr+sftp, bzr+ftp, bzr+lp, bzr+file, git+http, git+https, git+ssh, git+git, git+file, hg+file, hg+http, hg+https, hg+ssh, hg+static-http, svn+ssh, svn+http, svn+https, svn+svn, svn+file).
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

from IPython.display import Image, display

from lejepa_cotton import (
    DetectionEvalConfig,
    PretrainConfig,
    ProbeEvalConfig,
    WeightSources,
    run_detection_evaluation,
    run_pretraining,
    run_probe_evaluation,
)

ModuleNotFoundError: No module named 'lejepa_cotton'

## 1. Paths
The only place directories are defined. Edit to match your machine.

In [ ]:
MULTI_CAMERA_ROOT = Path("/Users/akintanoreofeoluwa/Downloads/LeJEPA _pretrainining_multi_camera_boll/mars_multi_camera_boll")
DETECTION_DATA = Path("image_dataset")          # contains images/ and labels/
PLOT_STATUS_DIR = Path("Plot Status")           # contains images and annotations.json
OUTPUT_ROOT = Path("outputs")

## 2. Cross-camera GAP LeJEPA pretraining
`max_samples` counts synchronised frame **triplets** (one image per camera), so 16,667 triplets is about 50k images. PCA snapshots are written at the first, middle and last epoch, coloured by camera: well-mixed colours mean viewpoint-invariant embeddings.

In [ ]:
pretrain_cfg = PretrainConfig(
    image_root=MULTI_CAMERA_ROOT,
    output_dir=OUTPUT_ROOT / "pretraining",
    cameras=(1, 2, 4),        # 2nd, 3rd and 5th physical cameras (0-indexed in file names)
    max_samples=16_667,
    views_per_camera=1,
    image_size=128,
    batch_size=16,
    epochs=60,
    proj_dim=128,
    lr=1e-3,
    weight_decay=1e-4,
    lam=0.2,
    model_cfg="yolov8n.yaml",  # empty weights
)

checkpoint, history = run_pretraining(pretrain_cfg)
history.tail()

In [ ]:
display(Image(filename=str(pretrain_cfg.output_dir / "plots" / "loss_curves.png")))
print("Interactive PCA snapshots:", *sorted((pretrain_cfg.output_dir / "plots" / "pca_3d").glob("*.html")), sep="\n")

## 3. Evaluation A - cotton boll detection fine-tuning (LeJEPA vs COCO)
Both variants are fine-tuned on the **same** train/val split with identical hyper-parameters. Add `"coco_backbone"` (COCO backbone, random neck/head - the like-for-like control) or `"scratch"` to `variants` for extra baselines.

In [ ]:
weights = WeightSources(lejepa_checkpoint=checkpoint, model_cfg="yolov8n.yaml", coco_weights="yolov8n.pt")

detection_cfg = DetectionEvalConfig(
    source_dir=DETECTION_DATA,
    output_dir=OUTPUT_ROOT / "detection",
    weights=weights,
    class_names=("cotton_boll",),
    variants=("lejepa", "coco"),
    image_size=256,
    batch_size=16,
    epochs=40,
    lr0=0.002,
    device="cpu",             # failsafe; use "mps" or "cuda" if stable on your machine
)

detection_summary = run_detection_evaluation(detection_cfg)
detection_summary

In [ ]:
display(Image(filename=str(detection_cfg.output_dir / "plots" / "detection_metrics.png")))
display(Image(filename=str(detection_cfg.output_dir / "plots" / "detection_curves.png")))
print("Box overlays (green = ground truth, red = prediction):", detection_cfg.output_dir / "overlays")

## 4. Evaluation B - plot-status linear probe on frozen GAP features (LeJEPA vs COCO)
The backbone is frozen (including BatchNorm statistics); only a linear layer is trained on the concatenated P3/P4/P5 GAP vectors.

In [ ]:
probe_cfg = ProbeEvalConfig(
    image_dir=PLOT_STATUS_DIR,
    annotation_path=PLOT_STATUS_DIR / "annotations.json",
    output_dir=OUTPUT_ROOT / "plot_status_probe",
    weights=weights,
    label_mapping={"headland": 0, "between_plots": 1, "in_plot": 2},
    variants=("lejepa", "coco"),
    scales=("P3", "P4", "P5"),
    image_size=256,
    epochs=50,
    test_ratio=0.5,
)

probe_summary = run_probe_evaluation(probe_cfg)
probe_summary

In [ ]:
probe_plots = probe_cfg.output_dir / "plots"
display(Image(filename=str(probe_plots / "probe_metrics.png")))
display(Image(filename=str(probe_plots / "probe_loss_curves.png")))
for variant in probe_cfg.variants:
    display(Image(filename=str(probe_plots / f"confusion_{variant}.png")))
    display(Image(filename=str(probe_plots / f"pca2d_{variant}.png")))